# RAG Evaluation Framework — Part 1 Offline Eval (Colab)

Runs the full Part 1 pipeline against the `RAG_LangChain_Demo` system: golden dataset -> retrieval + generation -> RAGAS + custom LLM judge -> rubric scoring -> calibration -> bottom-up failure taxonomy -> final report.

This notebook only *drives* the code in `src/eval/` — it doesn't reimplement any logic. Every cell below is equivalent to a documented CLI command in the repo's [README](https://github.com/Charu1806/RAG_Eval_Framework#running-the-pipeline); if a widget cell misbehaves, you can always fall back to the CLI commands shown in its markdown.

**Cost/time note:** the offline eval makes ~5-6 LLM calls per golden item (1 generation + 4 RAGAS metric calls + 1 custom judge call) across 42 items -- real API usage, not free. Use `LIMIT` below to smoke-test on a few items first.

## 1. Clone the repo and install dependencies

In [ ]:
import os

REPO_DIR = "/content/RAG_Eval_Framework"
if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules https://github.com/Charu1806/RAG_Eval_Framework.git {REPO_DIR}
else:
    print("Repo already cloned, pulling latest")
    !cd {REPO_DIR} && git pull && git submodule update --init --recursive
%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt -r rag_pipeline/config/requirements.txt

## 2. Set API keys

Preferred: Colab Secrets (left sidebar -> key icon) -> add `MISTRAL_API_KEY`, and optionally `OPENAI_API_KEY` / `ANTHROPIC_API_KEY` as judge fallbacks. This cell falls back to an interactive password prompt if a secret isn't found (works outside Colab too).

In [ ]:
import os, getpass

def set_key(env_var, required=True):
    if os.environ.get(env_var):
        return
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(env_var)
    except Exception:
        pass
    if not value and required:
        value = getpass.getpass(f"{env_var} (not found in Colab secrets) -- paste it here: ")
    if value:
        os.environ[env_var] = value

set_key("MISTRAL_API_KEY", required=True)   # generator (rag_pipeline) + default judge
set_key("OPENAI_API_KEY", required=False)   # judge fallback #2, optional
set_key("ANTHROPIC_API_KEY", required=False)  # judge fallback #3, optional
print("Keys set:", [k for k in ("MISTRAL_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY") if os.environ.get(k)])

## 3. Run the offline evaluation

Equivalent CLI: `python -m src.eval.offline_eval` (or `--limit N` for a smoke test). Set `LIMIT = None` below to run the full 42-item golden dataset.

In [ ]:
LIMIT = 5  #@param {type:"raw"}

limit_flag = f"--limit {LIMIT}" if LIMIT else ""
!python -m src.eval.offline_eval {limit_flag}

In [ ]:
import json
from pathlib import Path

REPO_ROOT = Path.cwd()
with open(REPO_ROOT / "reports" / "offline_eval_v1.json", encoding="utf-8") as f:
    offline_eval = json.load(f)

agg = offline_eval["aggregate"]
print(f"Scored {agg['n_items']} items -- overall rubric average: {agg['overall_average']:.2f} / 3.0")
print("Quadrant counts:", agg["quadrant_counts"])

## 4. Charts

In [ ]:
import matplotlib.pyplot as plt

dims = list(agg["rubric_averages"].keys())
values = list(agg["rubric_averages"].values())

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(dims, values, color="#4C72B0")
ax.axhline(agg["overall_average"], color="gray", linestyle="--", label=f"Overall avg ({agg['overall_average']:.2f})")
ax.set_ylim(0, 3)
ax.set_ylabel("Average score (0-3)")
ax.set_title("Rubric scores by dimension")
ax.legend()
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.05, f"{v:.2f}", ha="center")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**2x2 quadrant scatter.** The quadrant label (`true_quality` / `shipped_luck` / `doom_loop` / `honest_failure`) for each item comes from `rubric.classify_quadrant()`'s boolean `retrieval_correct` x `answer_correct` logic, *not* from these axes directly. The axes below are a continuous proxy built from the rubric sub-scores purely so the four categories are visually separable -- x is the retrieval-side rubric average (`context_precision`, `context_recall`), y is the answer-side average (`faithfulness`, `answer_relevancy`, `citation_accuracy`).

In [ ]:
quadrant_colors = {
    "true_quality": "#2ca02c",
    "shipped_luck": "#ff7f0e",
    "doom_loop": "#d62728",
    "honest_failure": "#7f7f7f",
}

xs, ys, colors = [], [], []
for it in offline_eval["items"]:
    r = it["rubric"]
    xs.append((r["context_precision"] + r["context_recall"]) / 2)
    ys.append((r["faithfulness"] + r["answer_relevancy"] + r["citation_accuracy"]) / 3)
    colors.append(quadrant_colors[it["quadrant"]])

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(xs, ys, c=colors, s=60, alpha=0.85, edgecolors="black", linewidths=0.5)
ax.axvline(1.5, color="gray", linestyle=":")
ax.axhline(1.5, color="gray", linestyle=":")
ax.set_xlim(-0.2, 3.2)
ax.set_ylim(-0.2, 3.2)
ax.set_xlabel("Retrieval-side rubric avg (context_precision, context_recall)")
ax.set_ylabel("Answer-side rubric avg (faithfulness, answer_relevancy, citation_accuracy)")
ax.set_title("Retrieval vs. answer quality per item\n(color = actual quadrant classification)")
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=10, label=q)
           for q, c in quadrant_colors.items()]
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## 5. Judge calibration (hand-labeling)

Blind by design: you label from the question/answer/retrieved-context, without seeing the judge's scores, so you don't anchor on them. This reuses `calibration.py`'s exact sampling and CSV schema, so `compute_calibration()` and the CLI `compare` command both work unchanged on whatever this cell saves.

CLI equivalent: `python -m src.eval.calibration template` (produces the same CSV for you to fill in with a text editor instead), then `... compare`.

**If the widgets below don't render or misbehave**, skip to the CLI fallback cell just after this section instead -- it does the same thing without any widget code.

In [ ]:
import csv
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

from src.eval.calibration import (
    sample_items_for_calibration, write_rubric_reference,
    RUBRIC_FIELDS, BOOLEAN_FIELDS, TEMPLATE_FIELDS, HUMAN_FIELDS,
)
from src.eval.offline_eval import load_golden_dataset

N_CALIBRATION_ITEMS = 18  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}

golden = load_golden_dataset()
invariants_by_id = {g["id"]: g["invariants"] for g in golden}
sampled = sample_items_for_calibration(offline_eval, n=N_CALIBRATION_ITEMS, seed=SEED)
write_rubric_reference()  # reports/calibration_rubric_reference.md -- open it alongside this cell
print(f"Sampled {len(sampled)} items for calibration. Rubric reference: reports/calibration_rubric_reference.md")

In [ ]:
state = {"index": 0, "labels": {item["id"]: {} for item in sampled}}

rubric_widgets = {name: widgets.IntSlider(min=0, max=3, description=name, style={"description_width": "140px"})
                  for name in RUBRIC_FIELDS}
bool_widgets = {name: widgets.Checkbox(description=name) for name in BOOLEAN_FIELDS}
notes_w = widgets.Textarea(description="notes", layout=widgets.Layout(width="100%"),
                            style={"description_width": "140px"})

content_out = widgets.Output()
progress_out = widgets.Output()


def render_item(idx):
    item = sampled[idx]
    saved = state["labels"].get(item["id"], {})
    with content_out:
        clear_output(wait=True)
        context_block = "\n\n---\n\n".join(item["retrieved_contexts"])
        invariants_block = "\n".join(f"- {inv}" for inv in invariants_by_id.get(item["id"], []))
        display(Markdown(
            f"### Item {idx + 1}/{len(sampled)} -- `{item['id']}` ({item['category']}, {item['difficulty']})\n\n"
            f"**Question:** {item['question']}\n\n"
            f"**Answer:** {item['answer']}\n\n"
            f"**Retrieved context:**\n\n{context_block}\n\n"
            f"**Required facts (invariants):**\n{invariants_block}"
        ))
    for name, w in rubric_widgets.items():
        w.value = saved.get(name, 0)
    for name, w in bool_widgets.items():
        w.value = saved.get(name, False)
    notes_w.value = saved.get("notes", "")
    with progress_out:
        clear_output(wait=True)
        labeled = sum(1 for v in state["labels"].values() if v)
        print(f"Labeled so far: {labeled}/{len(sampled)}")


def save_current():
    item = sampled[state["index"]]
    state["labels"][item["id"]] = {
        **{name: w.value for name, w in rubric_widgets.items()},
        **{name: w.value for name, w in bool_widgets.items()},
        "notes": notes_w.value,
    }


def on_prev(_):
    save_current()
    if state["index"] > 0:
        state["index"] -= 1
        render_item(state["index"])


def on_next(_):
    save_current()
    if state["index"] < len(sampled) - 1:
        state["index"] += 1
        render_item(state["index"])


def on_save_all(_):
    save_current()
    missing = [iid for iid, v in state["labels"].items() if not v]
    if missing:
        print(f"Not all items labeled yet, missing: {missing}")
        return
    out_path = REPO_ROOT / "reports" / "calibration_labels.csv"
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=TEMPLATE_FIELDS + HUMAN_FIELDS)
        writer.writeheader()
        for item in sampled:
            labels = state["labels"][item["id"]]
            writer.writerow({
                "id": item["id"], "category": item["category"], "difficulty": item["difficulty"],
                "question": item["question"], "answer": item["answer"],
                "retrieved_contexts": " ||| ".join(item["retrieved_contexts"]),
                "golden_invariants": "; ".join(invariants_by_id.get(item["id"], [])),
                **{f"human_{k}": v for k, v in labels.items() if k != "notes"},
                "human_notes": labels.get("notes", ""),
            })
    print(f"Saved {len(sampled)} hand labels to {out_path}")


prev_btn = widgets.Button(description="< Previous")
next_btn = widgets.Button(description="Next >")
save_btn = widgets.Button(description="Save all labels", button_style="success")
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
save_btn.on_click(on_save_all)

render_item(0)
display(
    content_out,
    widgets.HBox(list(rubric_widgets.values())),
    widgets.HBox(list(bool_widgets.values())),
    notes_w,
    widgets.HBox([prev_btn, next_btn, save_btn]),
    progress_out,
)

**CLI fallback**, if the widgets above didn't work for you -- run these in a terminal (or via `!` cells here), editing the CSV with any text editor/spreadsheet app in between:

```bash
python -m src.eval.calibration template   # writes reports/calibration_template.csv
#   ... open it, fill in the human_* columns, save as reports/calibration_labels.csv ...
python -m src.eval.calibration compare    # writes reports/calibration_report.json
```

In [ ]:
from src.eval.calibration import compute_calibration

calibration_result = compute_calibration()
print(f"Labeled items: {calibration_result['n_labeled']}")
print("Cohen's kappa by dimension:")
for dim, k in calibration_result["kappa"].items():
    print(f"  {dim}: {k}")

import json as _json
from pathlib import Path as _Path
with open(_Path("reports") / "calibration_report.json", "w", encoding="utf-8") as f:
    _json.dump(calibration_result, f, indent=2)

## 6. Failure taxonomy

Equivalent CLI: `python -m src.eval.failure_taxonomy` (`--no-llm-labels` to skip the LLM-generated cluster names and use TF-IDF top terms only, no extra API calls).

In [ ]:
!python -m src.eval.failure_taxonomy

In [ ]:
with open(REPO_ROOT / "reports" / "failure_taxonomy.json", encoding="utf-8") as f:
    taxonomy = json.load(f)

if taxonomy["n_failed"] == 0:
    print("No failures in this run -- nothing to chart.")
else:
    clusters = taxonomy["clusters"]
    labels = [c["label"] for c in clusters]
    sizes = [c["size"] for c in clusters]

    fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(clusters))))
    ax.barh(labels[::-1], sizes[::-1], color="#c44e52")
    ax.set_xlabel("Number of failed items")
    ax.set_title(
        f"Failure taxonomy -- {taxonomy['n_failed']}/{taxonomy['n_total_items']} items "
        f"failed ({taxonomy['failure_rate']:.0%})"
    )
    plt.tight_layout()
    plt.show()

## 7. Generate the final report

Combines everything above into `reports/offline_eval_v1.md` and updates the repo README's Results section. Equivalent CLI: `python -m src.eval.generate_report`.

In [ ]:
!python -m src.eval.generate_report

In [ ]:
from IPython.display import Markdown, display

with open(REPO_ROOT / "reports" / "offline_eval_v1.md", encoding="utf-8") as f:
    display(Markdown(f.read()))

## Next steps

- Commit `reports/*.json`, `reports/offline_eval_v1.md`, and the updated `README.md` back to the repo so results are versioned like the code (see the main [README](https://github.com/Charu1806/RAG_Eval_Framework)).
- Part 2 (traffic simulation) and Part 3 (closing the loop) aren't built yet -- see [PRD.md](https://github.com/Charu1806/RAG_Eval_Framework/blob/main/PRD.md).